# System-Level Optimizations

Test infrastructure-level optimizations for the most promising serving option(s).

Optimizations covered:
1. **Instance scaling** -- vary number of model instances on GPU
2. **Dynamic batching** -- configure Triton dynamic batching
3. **Right-sizing** -- measure CPU/memory/GPU usage under load

These tests are run on the Chameleon `gpu_rtx_6000` node using Triton Inference Server.

In [ ]:
import subprocess
import time
import pandas as pd

TRITON_URL = "triton_server:8000"
INPUT_DATA = "/workspace/triton/input_data_text.json"

# Set this to your most promising ONNX model
BEST_MODEL = "distilbert_classifier"
BEST_MODEL_PT = "distilbert_classifier_pt"

optimization_results = []

In [ ]:
def run_perf(model_name, extra_args=""):
    cmd = (
        f"perf_analyzer -u {TRITON_URL} "
        f"-m {model_name} "
        f"--input-data {INPUT_DATA} "
        f"{extra_args}"
    )
    print(f"CMD: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.stderr:
        print(f"STDERR (last 500): {result.stderr[-500:]}")
    return result.stdout

## 1. Instance Scaling

Test different `instance_group` counts in `config.pbtxt`.

**Procedure**: For each configuration, update `config.pbtxt`, restart Triton,
then run perf_analyzer.

### Configurations to test:

```
# Config A: 1 instance on GPU 0 (baseline)
instance_group [{ count: 1, kind: KIND_GPU, gpus: [0] }]

# Config B: 2 instances on GPU 0
instance_group [{ count: 2, kind: KIND_GPU, gpus: [0] }]

# Config C: 4 instances on GPU 0
instance_group [{ count: 4, kind: KIND_GPU, gpus: [0] }]
```

After each config change, run:
```bash
# On host:
docker compose -f docker/docker-compose-triton.yaml restart triton_server
docker logs triton_server  # Verify model is READY
```

In [ ]:
instance_configs = {
    "1_instance": "Baseline: 1 GPU instance",
    "2_instances": "2 GPU instances",
    "4_instances": "4 GPU instances",
}

for config_name, description in instance_configs.items():
    print(f"\n{'='*60}")
    print(f"Instance Scaling: {description}")
    print(f"{'='*60}")
    print(f"Make sure config.pbtxt is set for {config_name} and Triton is restarted.")
    print(f"Then run the cell below.")

In [ ]:
# Run after setting 1 instance config
print("=== 1 Instance Benchmark ===")
run_perf(BEST_MODEL_PT, "--concurrency-range 1:8:1 -b 1")

In [ ]:
# Run after setting 2 instance config
print("=== 2 Instances Benchmark ===")
run_perf(BEST_MODEL_PT, "--concurrency-range 1:8:1 -b 1")

In [ ]:
# Run after setting 4 instance config
print("=== 4 Instances Benchmark ===")
run_perf(BEST_MODEL_PT, "--concurrency-range 1:8:1 -b 1")

## 2. Dynamic Batching

Add dynamic batching to `config.pbtxt`:

```
dynamic_batching {
  preferred_batch_size: [4, 8, 16]
  max_queue_delay_microseconds: 100
}
```

Then restart Triton and run benchmarks.

In [ ]:
# Verify dynamic batching is enabled
import requests

resp = requests.get(f"http://{TRITON_URL}/v2/models/{BEST_MODEL_PT}/versions/1/stats")
if resp.status_code == 200:
    stats = resp.json()
    print(f"Model stats: {stats}")

In [ ]:
print("=== Dynamic Batching: Concurrency Sweep ===")
run_perf(BEST_MODEL_PT, "--concurrency-range 1:16:1 -b 1")

In [ ]:
print("=== Dynamic Batching: Request Rate (Constant) ===")
run_perf(BEST_MODEL_PT, "--request-rate-range 10:200:20 --request-distribution constant -b 1")

In [ ]:
print("=== Dynamic Batching: Request Rate (Poisson) ===")
run_perf(BEST_MODEL_PT, "--request-rate-range 10:200:20 --request-distribution poisson -b 1")

### Test different batch configurations

In [ ]:
batch_configs = [
    {"name": "small_batch", "preferred": "[2, 4]", "delay": 100},
    {"name": "medium_batch", "preferred": "[4, 8, 16]", "delay": 100},
    {"name": "large_batch", "preferred": "[8, 16, 32]", "delay": 500},
    {"name": "aggressive_batch", "preferred": "[4, 8, 16]", "delay": 1000},
]

print("For each batch config:")
print("1. Update config.pbtxt with the dynamic_batching settings")
print("2. Restart Triton")
print("3. Run: perf_analyzer -u triton_server:8000 -m <model> --input-data input.json --concurrency-range 1:16")
print()
for cfg in batch_configs:
    print(f"--- {cfg['name']} ---")
    print(f"dynamic_batching {{")
    print(f"  preferred_batch_size: {cfg['preferred']}")
    print(f"  max_queue_delay_microseconds: {cfg['delay']}")
    print(f"}}")
    print()

## 3. Right-Sizing

Measure resource usage under representative load for the most promising option.

In [ ]:
print("=== GPU Utilization ===")
result = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
print(result.stdout)

In [ ]:
print("=== Docker Container Stats ===")
result = subprocess.run(
    "docker stats --no-stream --format 'table {{.Name}}\t{{.CPUPerc}}\t{{.MemUsage}}\t{{.MemPerc}}'  2>/dev/null || echo 'docker stats not available from inside container'",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

In [ ]:
print("=== GPU Memory During Load Test ===")
print("Run the following in a parallel terminal while perf_analyzer is active:")
print()
print("  nvidia-smi --query-gpu=timestamp,name,utilization.gpu,utilization.memory,memory.used,memory.total --format=csv -l 1")
print()
print("Or use nvtop for real-time monitoring.")

## 4. Combined Optimizations

Test the best combination: ONNX model + dynamic batching + multiple instances.

In [ ]:
print("=== Combined: ONNX + Dynamic Batching + 2 Instances ===")
print("Config:")
print("  backend: onnxruntime")
print("  instance_group [{ count: 2, kind: KIND_GPU, gpus: [0] }]")
print("  dynamic_batching { preferred_batch_size: [4, 8, 16], max_queue_delay_microseconds: 100 }")
print()
print("After updating config and restarting Triton:")
run_perf(BEST_MODEL, "--concurrency-range 1:16:1 -b 1 --shape input_ids:64 --shape attention_mask:64")

## Summary

Record observations for each configuration in the serving options table:

| Configuration | Throughput (infer/sec) | p50 Latency (ms) | p99 Latency (ms) | GPU Util (%) | GPU Memory (MB) |
|---|---|---|---|---|---|
| 1 instance, no batching | ... | ... | ... | ... | ... |
| 2 instances, no batching | ... | ... | ... | ... | ... |
| 4 instances, no batching | ... | ... | ... | ... | ... |
| 1 instance, dynamic batching (medium) | ... | ... | ... | ... | ... |
| 2 instances, dynamic batching (medium) | ... | ... | ... | ... | ... |
| ONNX + 2 instances + dynamic batching | ... | ... | ... | ... | ... |